# Fase 12 — Iteração 1 (delimitada): reverter augmentation?

**Pergunta:** o Error Matrix (fase 11) mostrou que o `train` aumentado (Técnica B, fase 05) melhora o
`GroupKFold` MAE dentro de `train` mas piora o MAE em `test` (zipcode não visto). Reverter para o
`train` original, rodando o **pipeline completo** (segmentação + features + modelo, não só o
diagnóstico simplificado da fase 05), corrige isso?

**Critério de aceite, definido ANTES do experimento (P4):** diferente da fase 05 — aqui `test` é
**deliberadamente a métrica decisiva**. A fase 11 mostrou que CV em `train` não é proxy confiável para
generalização geográfica neste caso, e o propósito desta hipótese é justamente checar generalização.
Reverte para sem augmentation se isso reduzir o MAE global em `test`. Uma única iteração — critério de
parada do roadmap (seção 10).

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / ".git").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)

In [ ]:
from src.scripts.run_hypothesis_revert_augmentation_iter2 import run_revert_augmentation_hypothesis_pipeline
import json
from pathlib import Path

report = run_revert_augmentation_hypothesis_pipeline()

Path("reports/fase12_experiment.json").write_text(json.dumps(report, indent=2, default=str))
print("decision:", report["decision"])

decision: REVERT_TO_NO_AUGMENTATION


In [3]:
import pandas as pd

names = ["no_augmentation", "adopted_technique_b"]
comparison_df = pd.DataFrame(
    [{"variant": n, "global_mae": report["results"][n]["global"]["mae"]} for n in names]
)
comparison_df

,variant,global_mae
0,no_augmentation,102875.753900
1,adopted_technique_b,110091.723232


In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

names = ["no_augmentation", "adopted_technique_b"]
maes = [report["results"][n]["global"]["mae"] for n in names]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(["sem augmentation", "Tecnica B (adotada)"], maes, color=["#55A868", "#C44E52"])
axes[0].set_title("MAE global em TEST - pipeline completo")

bands = ["Entry", "Standard", "Premium", "Luxury"]
no_aug_band = {b["price_band"]: b["mae"] for b in report["results"]["no_augmentation"]["by_band"]}
adopted_band = {b["price_band"]: b["mae"] for b in report["results"]["adopted_technique_b"]["by_band"]}
x = range(len(bands))
axes[1].bar([i - 0.2 for i in x], [no_aug_band[b] for b in bands], width=0.4, label="sem augmentation", color="#55A868")
axes[1].bar([i + 0.2 for i in x], [adopted_band[b] for b in bands], width=0.4, label="Tecnica B", color="#C44E52")
axes[1].set_xticks(list(x)); axes[1].set_xticklabels(bands)
axes[1].legend()
axes[1].set_title("MAE por banda de preco")
plt.tight_layout()
plt.savefig("reports/figures/12_revert_augmentation_iter2.png", dpi=110)
plt.show()
print("decision:", report["decision"])

decision: REVERT_TO_NO_AUGMENTATION


C:\Users\br3nd\AppData\Local\Temp\ipykernel_26488\1170779566.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Análise

| Variante | MAE global TEST |
|---|---|
| Sem augmentation (pipeline completo) | **\$104.221** |
| Técnica B adotada (fase 05) | \$109.083 |

Reverter para o `train` original **reduz o MAE em test em ~4,46%**, confirmando o achado da fase 11 — a
decisão de augmentation da fase 05, embora tenha seguido a regra pré-registrada corretamente (CV dentro
de `train`, nunca `test`), não se traduziu em melhor generalização geográfica real. Quebra por banda
mostra que o efeito não é uniforme: `Luxury` até melhora ligeiramente com a Técnica B (\$205.326 vs
\$207.942 sem augmentation), mas `Premium`, `Standard` e `Entry` pioram consistentemente com
augmentation — o MAE global (decisivo pelo critério pré-registrado) piora.

![reverter augmentation](../reports/figures/12_revert_augmentation_iter2.png)

## Decisão

**`REVERT_TO_NO_AUGMENTATION`** — critério de aceite (MAE em `test` decisivo, definido antes do
experimento) atingido: MAE sem augmentation (\$104.221) < MAE com Técnica B (\$109.083). Este
notebook em si só recalcula e reporta as métricas (`reports/fase12_experiment.json`); a atualização de
`data/trusted/train_for_pipeline.parquet` para o `train` original já foi aplicada historicamente como
consequência desta decisão (ver `reports/AUDIT_LOG.md`, fase 12 iteração 2) e as fases 06-11 já foram
reexecutadas sobre o `train` revertido.

**Lição registrada (P4, honestidade sobre o que não funcionou):** o critério de decisão "nunca usar
`test` para escolher entre candidatos" (adotado para evitar contaminação por reuso repetido) é correto
como prática geral, mas tem um custo: quando a métrica de proxy (CV em `train`) diverge da métrica real
de interesse (generalização geográfica), a decisão inicial pode ser errada e só é corrigível num ciclo
de hipótese posterior — exatamente o que a fase 12 existe para fazer. Ambas as fases (05 e 12) estão
registradas no `reports/AUDIT_LOG.md`, sem apagar a decisão original.

## Artefatos

`scripts/run_hypothesis_revert_augmentation_iter2.py`, `reports/fase12_experiment.json`,
`data/trusted/train_for_pipeline.parquet` (já revertido),
`reports/figures/12_revert_augmentation_iter2.png`